In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("LSTM", exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:{}".format(device))

# 定义 LSTM 模型
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.fc(out)
        return out

device:cpu


In [2]:
dff=pd.read_csv("dff.csv",header=0,index_col=0)
dff=dff.fillna(0)

In [3]:
dff1= pd.read_csv("Data_raw.csv",header=0,index_col=0)

In [4]:
dff["STR"] = dff1["STR"];

In [5]:
L=245 # 需要训练245个Reservoir
Total = 816 #总共有816个数据
#For entire sample forecasting (1997.08-2017.12)
WL=Total-L # rolling window length 训练每一个Reservoir的数据集大小 WL=windows_length
ws=0# rolling window start index ws=277 for subsample, ws=0 for entire sample

In [6]:
RV=np.array(dff['RV'],dtype=np.float32).reshape(-1,1)

In [7]:
F = ["RV", "MKT", "diff_DP", "IP", "DEF","EP","SMB","diff_TB", "HML", "INF","STR"]
features = len(F)

In [8]:
Data=np.array(dff[F],dtype=np.float32).reshape(-1,features)

In [9]:
test = dff.iloc[-245:]

In [10]:
# Shared hyperparameters (hidden_size and input_size vary per run)
num_layers = 2
output_size = 1
num_epochs = 100
batch_size = 256
learning_rate = 0.001
K = 3

In [ ]:
def train_model(model, train_loader, optimizer, criterion):
    for epoch in range(num_epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model

def run_lstm_rolling(feature_list, hidden_size, K=3):
    """Run the full 245-window rolling LSTM and return predictions."""
    n_feat = len(feature_list)
    data_arr = np.array(dff[feature_list], dtype=np.float32).reshape(-1, n_feat)
    
    predictions = np.zeros((L, 1))
    ws = 0
    while ws < L:
        model = LSTMModel(n_feat, hidden_size, num_layers, output_size).to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()

        x_train = torch.zeros(WL - K, K, n_feat)
        y_train = torch.from_numpy(RV[ws + K:ws + WL])
        
        for t in range(WL - K):
            X = np.array(data_arr[ws + t:ws + t + K, :]).reshape(-1, n_feat)
            x_train[t, :, :] = torch.from_numpy(X)
        train_dataset = TensorDataset(x_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        model = train_model(model, train_loader, optimizer, criterion)
        
        x_test = torch.zeros(1, K, n_feat).to(device)
        x_test[0, :, :] = torch.from_numpy(np.array(data_arr[ws + WL - K:ws + WL]).reshape(-1, n_feat))
        
        model.eval()
        with torch.no_grad():
            predictions[ws, 0] = model(x_test)
        
        ws += 1
        if ws % 50 == 0:
            print(f"  Window {ws}/{L}")
    return predictions

In [12]:
# --- Run 1: Base LSTM (RV only, hidden_size=60) ---
F_base = ["RV"]
print("Training LSTM (base, RV-only, hidden=60) ...")
predictions_lstm = run_lstm_rolling(F_base, hidden_size=60)
np.savetxt("LSTM/lstm60_predictions.csv", predictions_lstm, delimiter=',')
print("Saved LSTM/lstm60_predictions.csv")
print("MSE:", np.mean((predictions_lstm - RV[WL:]) ** 2))

Training LSTM (base, RV-only, hidden=60) ...


NameError: name 'criterion' is not defined

In [ ]:
# --- Run 2: LSTMX (all 11 features, hidden_size=50) ---
F_ext = ["RV", "MKT", "diff_DP", "IP", "DEF", "EP", "SMB", "diff_TB", "HML", "INF", "STR"]
print("Training LSTMX (extended, 11 features, hidden=50) ...")
predictions_lstmx = run_lstm_rolling(F_ext, hidden_size=50)
np.savetxt("LSTM/lstmx50_predictions.csv", predictions_lstmx, delimiter=',')
print("Saved LSTM/lstmx50_predictions.csv")
print("MSE:", np.mean((predictions_lstmx - RV[WL:]) ** 2))

In [ ]:
plt.figure(figsize=(10, 6))
plt.rcParams['font.size'] = 16
plt.title("LSTM vs LSTMX - Predicted vs Actual RV")
plt.xlabel("$Date$")
plt.plot(test.index, predictions_lstm, label="LSTM (RV, h=60)", color="red")
plt.plot(test.index, predictions_lstmx, label="LSTMX (11feat, h=50)", color="green")
plt.plot(test.index, RV[WL:], label="Actual", color="blue")
plt.xticks(test.index[::24], rotation=-45)
plt.ylabel('Realized Volatility')
plt.tight_layout() 
plt.legend()
plt.savefig("LSTM/lstm_comparison.png")
plt.show()

In [ ]:
def compute_qlike(forecasts, actuals):
    """
    Compute the QLIKE (Quasi-Likelihood) loss function for evaluating forecasting accuracy.
    forecasts: Forecasted variance (sigma squared from a model)
    actuals: Realized variance (actual observed variance)
    """
    # Using absolute values of forecasts and actuals
    forecasts = np.abs(forecasts)
    actuals = np.abs(actuals)

    # Calculate the ratio and ensure it's positive
    ratio = actuals / forecasts

    # Compute QLIKE
    qlike = np.sum(ratio - np.log(ratio) - 1)
    return qlike


def qlike_loss(y_true, y_hat):
  """
  This function computes the mean Quasi-Likelihood (QLIKE) loss function.

  Args:
      y_true (np.ndarray): True values of the variable.
      y_hat (np.ndarray): Predicted values of the variable.

  Returns:
      float: Mean QLIKE loss between the true and predicted values.
  """
  eps = np.finfo(float).eps  # Machine epsilon for numerical stability
  w = np.abs(y_true - y_hat) / (y_hat + eps)
  return np.mean(np.log(1 + w**2))

In [ ]:
actual = RV[WL:]

In [ ]:
print("--- LSTM (base, RV-only, hidden=60) ---")
qlike1 = qlike_loss(np.array(actual), np.array(predictions_lstm))
print(f"  QLIKE loss: {qlike1}")
mse1 = np.mean((predictions_lstm - actual) ** 2)
print(f"  MSE: {mse1}")

print("\n--- LSTMX (extended, 11 features, hidden=50) ---")
qlike2 = qlike_loss(np.array(actual), np.array(predictions_lstmx))
print(f"  QLIKE loss: {qlike2}")
mse2 = np.mean((predictions_lstmx - actual) ** 2)
print(f"  MSE: {mse2}")